In [1]:
import deepchem as dc
import numpy as np
import pandas as pd
import os
import tempfile
import random
import warnings
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import glob
warnings.filterwarnings("ignore", category=UserWarning)

param_grid = {
    'hidden_layer_sizes': [(100,), (200,)],
    'alpha': [0.0001, 0.001]
}



#param_grid = {                         # for bigger qm8
#    'hidden_layer_sizes': [(100,), (200,)],
#    'alpha': [0.005,0.01]
#}


def df_to_ecfp_dataset(df_input, tasks):
    featurizer = dc.feat.CircularFingerprint(size=2048)
    loader = dc.data.CSVLoader(tasks=tasks, smiles_field='SMILES', featurizer=featurizer)
    temp_path = tempfile.mktemp(suffix=".csv")
    df_input.to_csv(temp_path, index=False)
    dataset = loader.create_dataset(temp_path)
    os.remove(temp_path)
    return dataset

def evaluate_mlp(model, dataset):
    y_true = dataset.y.flatten()
    y_pred = model.predict(dataset.X)
    return {
        'r2_score': r2_score(y_true, y_pred),
        'mae_score': mean_absolute_error(y_true, y_pred),
        'rms_score': mean_squared_error(y_true, y_pred, squared=False)
    }
all_results_sole = []
path = os.path.dirname(os.getcwd())
train_df = pd.read_csv(f'{path}/datasets/example_train.csv')
train_df_filtered = pd.read_csv(f'{path}/datasets/results/clean_3_3.csv').iloc[:,:2]       ## choose the filtered dataset you want to use.
valid_df = pd.read_csv(f'{path}/datasets/example_valid.csv')
test_df  = pd.read_csv(f'{path}/datasets/example_test.csv')
tasks = ['Labels']


train_dataset = df_to_ecfp_dataset(train_df, tasks)
train_filtered_dataset = df_to_ecfp_dataset(train_df_filtered, tasks)

valid_dataset = df_to_ecfp_dataset(valid_df, tasks)
test_dataset  = df_to_ecfp_dataset(test_df, tasks)

train_dataset_dict = {'original': train_dataset,'filtered': train_filtered_dataset}

all_results_sole = []

for dataset_type, current_train_dataset in train_dataset_dict.items():
    print(f"\n===== Training with {dataset_type} dataset =====")
    best_val_r2 = -np.inf
    best_model = None
    best_params = None
    for hidden_layer_sizes in param_grid['hidden_layer_sizes']:
        for alpha in param_grid['alpha']:
            model = MLPRegressor(
                hidden_layer_sizes=hidden_layer_sizes,
                alpha=alpha,
                max_iter=500
            )
            model.fit(
                current_train_dataset.X,
                current_train_dataset.y.flatten()
            )
            val_pred = model.predict(valid_dataset.X)
            val_r2 = r2_score(
                valid_dataset.y.flatten(),
                val_pred
            )
            print(
                f"→ MLP(hidden_layer_sizes={hidden_layer_sizes}, "
                f"alpha={alpha}) → Val R² = {val_r2:.4f}"
            )
            if val_r2 > best_val_r2:
                best_val_r2 = val_r2
                best_model = model
                best_params = {
                    'hidden_layer_sizes': hidden_layer_sizes,
                    'alpha': alpha
                }
    train_scores = evaluate_mlp(
        best_model,
        current_train_dataset
    )
    valid_scores = evaluate_mlp(
        best_model,
        valid_dataset
    )
    test_scores = evaluate_mlp(
        best_model,
        test_dataset
    )
    print(f"Dataset: {dataset_type}")
    print(f"Train: {train_scores}")
    print(f"Valid: {valid_scores}")
    print(f"Test : {test_scores}")
    all_results_sole.append({
        'dataset_type': dataset_type,
        'used_params': best_params,
        'train_scores': train_scores,
        'valid_scores': valid_scores,
        'test_scores': test_scores
    })
results_df = pd.DataFrame([{
    'dataset_type': res['dataset_type'],
    **res['used_params'],
    'train_r2': res['train_scores']['r2_score'],
    'train_mae': res['train_scores']['mae_score'],
    'train_rmse': res['train_scores']['rms_score'],
    'valid_r2': res['valid_scores']['r2_score'],
    'valid_mae': res['valid_scores']['mae_score'],
    'valid_rmse': res['valid_scores']['rms_score'],
    'test_r2': res['test_scores']['r2_score'],
    'test_mae': res['test_scores']['mae_score'],
    'test_rmse': res['test_scores']['rms_score']
    } for res in all_results_sole])

results_df.to_csv(f"{path}/datasets/results/mlp_results.csv",index=False)

smiles_field is deprecated and will be removed in a future version of DeepChem.Use feature_field instead.
smiles_field is deprecated and will be removed in a future version of DeepChem.Use feature_field instead.
smiles_field is deprecated and will be removed in a future version of DeepChem.Use feature_field instead.
smiles_field is deprecated and will be removed in a future version of DeepChem.Use feature_field instead.



===== Training with original dataset =====
→ MLP(hidden_layer_sizes=(100,), alpha=0.0001) → Val R² = 0.1524
→ MLP(hidden_layer_sizes=(100,), alpha=0.001) → Val R² = 0.1048
→ MLP(hidden_layer_sizes=(200,), alpha=0.0001) → Val R² = 0.1430
→ MLP(hidden_layer_sizes=(200,), alpha=0.001) → Val R² = 0.1094
Dataset: original
Train: {'r2_score': 0.9667723259759127, 'mae_score': 0.1756688606340914, 'rms_score': 0.851305506244957}
Valid: {'r2_score': 0.15240292188295212, 'mae_score': 2.8254547531707264, 'rms_score': 4.0598671117349445}
Test : {'r2_score': 0.24865856013408338, 'mae_score': 2.3749041709977634, 'rms_score': 3.547082692769426}

===== Training with filtered dataset =====
→ MLP(hidden_layer_sizes=(100,), alpha=0.0001) → Val R² = 0.1015
→ MLP(hidden_layer_sizes=(100,), alpha=0.001) → Val R² = 0.0942
→ MLP(hidden_layer_sizes=(200,), alpha=0.0001) → Val R² = 0.0989
→ MLP(hidden_layer_sizes=(200,), alpha=0.001) → Val R² = 0.0880
Dataset: filtered
Train: {'r2_score': 0.9637570757280635, 'm